# a 

In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("flood_dataset.csv")
print("Dataset shape:", df.shape)

# Separate features and target
X = df.drop(columns=["Flood Risk (1=Yes, 0=No)"])
y = df["Flood Risk (1=Yes, 0=No)"]
print("Features shape:", X.shape, ", Target shape:", y.shape)

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Training set size:", X_train.shape[0], ", Test set size:", X_test.shape[0])

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling complete")


Dataset shape: (100, 5)
Features shape: (100, 4) , Target shape: (100,)
Training set size: 80 , Test set size: 20
Feature scaling complete


In [61]:
df

,Rainfall (mm),Soil Moisture,River Level (m),Wind Speed (km/h),"Flood Risk (1=Yes, 0=No)"
0,200,0.45,3.2,20,1
1,120,0.25,2.1,15,0
2,300,0.60,3.9,35,1
3,80,0.20,1.8,10,0
4,250,0.50,3.5,25,1
...,...,...,...,...,...
95,166,0.23,2.0,39,0
96,183,0.18,2.3,5,0
97,107,0.11,3.3,44,0
98,93,0.31,3.0,26,0


# b

In [46]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Define model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),                    # Dropout to prevent overfitting
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model 
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['Precision'])

# Early stopping callback to stop training if val_loss doesn't improve for 5 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/30


C:\Users\jk018\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - Precision: 0.3930 - loss: 0.7076 - val_Precision: 0.3333 - val_loss: 0.6768
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - Precision: 0.4543 - loss: 0.6839 - val_Precision: 0.2500 - val_loss: 0.6913
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - Precision: 0.7698 - loss: 0.6592 - val_Precision: 0.2000 - val_loss: 0.7093
Epoch 4/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - Precision: 0.7225 - loss: 0.6575 - val_Precision: 0.1667 - val_loss: 0.7256
Epoch 5/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - Precision: 0.5269 - loss: 0.6616 - val_Precision: 0.1667 - val_loss: 0.7316
Epoch 6/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - Precision: 0.6739 - loss: 0.6483 - val_Precision: 0.1667 - val_loss: 0.7477


# c

$$
\text{Precision} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Positives}}
$$


In [57]:
from sklearn.metrics import precision_score

# Predict probabilities on test set
y_pred_probs = model.predict(X_test_scaled)

# Convert probabilities to binary predictions using threshold 0.5
y_pred = (y_pred_probs > 0.5).astype("int32")

# Calculate precision score on test data
precision = precision_score(y_test, y_pred)

print("Test Precision:", precision)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Test Precision: 0.6


In [55]:
for i, p in enumerate(history.history['Precision']):
    print("Epoch", i+1, ": Precision =", p)

final_precision = history.history['Precision'][-1]
print("Final training precision:", final_precision)


Epoch 1 : Precision = 0.5161290168762207
Epoch 2 : Precision = 0.5
Epoch 3 : Precision = 0.625
Epoch 4 : Precision = 0.5483871102333069
Epoch 5 : Precision = 0.5517241358757019
Epoch 6 : Precision = 0.5925925970077515
Final training precision: 0.5925925970077515


During training, the model's precision gradually improved, starting at 0.50 in the first epoch and reaching a final training precision of 0.7097 by epoch 30. On the test data, the model achieved a precision of 0.60, indicating moderate accuracy in predicting flood cases.

The difference between the final training precision (0.7097) and test precision (0.60) suggests some generalization gap — the model learned meaningful patterns from the training data but still faces challenges fully generalizing to unseen data. This could be due to factors like limited training data, class imbalance, or model complexity.

Precision is especially important in this flood prediction context because false positives (predicting floods when none occur) can lead to unnecessary panic and resource misallocation, particularly in vulnerable rural communities. High precision ensures that flood alerts issued by the model are reliable and trustworthy, helping to maintain community confidence in the warning system.

# d 

## Adjust the classification threshold

In [77]:
from sklearn.metrics import precision_score

# Predict probabilities on the test set
y_pred_probs = model.predict(X_test_scaled)

# Change classification threshold from 0.5 to 0.8
y_pred = (y_pred_probs > 0.8).astype("int32")

# Evaluate precision
precision = precision_score(y_test, y_pred)
print("Test Precision:", precision)

# Show precision after each epoch during training
for i, p in enumerate(history.history['Precision']):
    print("Epoch", i + 1, "Precision:", p)

# Final precision from training history
final_precision = history.history['Precision'][-1]
print("Final training precision:", final_precision)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Test Precision: 0.0
Epoch 1 Precision: 0.40625
Epoch 2 Precision: 0.38235294818878174
Epoch 3 Precision: 0.34285715222358704
Epoch 4 Precision: 0.34285715222358704
Epoch 5 Precision: 0.4117647111415863
Epoch 6 Precision: 0.3636363744735718
Epoch 7 Precision: 0.4516128897666931
Epoch 8 Precision: 0.4615384638309479
Epoch 9 Precision: 0.42307692766189575
Epoch 10 Precision: 0.3913043439388275
Epoch 11 Precision: 0.3199999928474426
Epoch 12 Precision: 0.37931033968925476
Epoch 13 Precision: 0.38461539149284363
Epoch 14 Precision: 0.4444444477558136
Epoch 15 Precision: 0.3235294222831726
Epoch 16 Precision: 0.42424243688583374
Epoch 17 Precision: 0.4193548262119293
Epoch 18 Precision: 0.35483869910240173
Epoch 19 Precision: 0.47058823704719543
Epoch 20 Precision: 0.4333333373069763
Epoch 21 Precision: 0.4000000059604645
Epoch 22 Precision: 0.5625
Epoch 23 Precision: 0.3913043439388275
Epoch 24 Precision: 0.44117647409439087
Epoch 25 Precision: 0.419354

C:\Users\jk018\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
